# 독립 감사와 수정된 점검 출력

판정은 **HYBRID**다. 기존 규범 기반 배정은 민감도 분석 층으로 보존하고,
실제 출력은 Q1/Q2/Q3 근거·자료 확인·불확실 공동 점검군으로 연결한다.
이 노트북은 저장된 실행 결과를 읽는다. 분석을 다시 실행하려면
`python src/run_independent_audit.py --replicates 400`을 사용한다.

CAI는 표집된 정책 파라미터의 비중이다. 위기 확률이 아니며, CAI=0이어도
연속공간에서 가능한 배정이 존재할 수 있다. 최신 보고서: `outputs/independent_audit/REPORT.md`.


In [1]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
OUT = ROOT / 'outputs/independent_audit'
def show(name, columns=None):
    frame = pd.read_csv(OUT / (name + '.csv'))
    print((frame if columns is None else frame[columns]).to_string(index=False))
print(json.dumps(json.loads((OUT/'summary.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=2))


{
  "rows": 180,
  "complete": 160,
  "discriminating": 89,
  "replicates": 400,
  "seed": 99,
  "gate_changes": 0,
  "history_g4_changed_rows": 40,
  "vrc4_duration_1_pass_count": 40,
  "vrc4_duration_2_pass_count": 899,
  "vrc4_duration_3_pass_count": 899,
  "vrc4_duration_4_pass_count": 899,
  "corner_qp_configurations_tried": 81,
  "continuous_certified_rows": 154,
  "continuous_unresolved_rows": 6,
  "sample_missed_categories": 5
}


## 1. 이전 지적을 코드와 자료로 다시 판정

In [2]:
show('findings', ['지적','판정','근거'])


             지적                  판정                                                                            근거
     RC/VRC 독립성           CONFIRMED     prereg AM3·AM4·AM6의 선행 결과 인지 기록. 다만 작성자의 인지 상태와 실제 등록 시각은 파일만으로 독립 검증 불가.
  VRC4 단일 규범 의존           CONFIRMED                                양립 표본 1745→447. 지속 2·3·4분기 모두 899/6000 통과로 동일.
     합격선의 운영 근거 PARTIALLY_CONFIRMED                                  합격선·개정 이력은 명시. 실제 담당자 비용·점검 용량에 따른 도출 근거 없음.
  I3/Jaccard 정합           CONFIRMED                I3 고정 6조합 일부는 양립공간 밖. C3c는 무제약 1000표본. 같은 최종 구간 검증으로 취급할 수 없음.
        필연배정 명칭           CONFIRMED                   표본 공통 배정을 필연/확정으로 표기. 실제 feasible witness가 표본 누락 배정 5건을 확인.
MRSort 추가 효용 부재 PARTIALLY_CONFIRMED                 count2와 실제 차이는 2/160행. 다만 중요성을 판정할 현장 정답은 없으므로 무용함까지 입증되지 않음.
    g1·g2·g4 중복 PARTIALLY_CONFIRMED                           공통 원계열이나 각각 배정을 바꾸는 행 존재. 규모/강도/지속 차원이 완전히 동일하지 않음.
       교란의 비현실성 PARTIALLY_CONFIRMED       g 자체 독립교란은 아님. 원 수준과 lag의 독립 복원추출로 같은 분기 두 값이 

## 2. 표본 배정과 연속공간 검증

미해결은 불가능 판정이 아니다. 보수적 외부범위와 발견한 실제 해를 구분한다.

In [3]:
cert = pd.read_csv(OUT/'continuous_space_certificates.csv').fillna('')
print(cert[cert.sample_possible != cert.witnessed_possible][['industry','quarter','sample_possible','witnessed_possible']].to_string(index=False))
print('\n미해결 행')
print(cert[cert.full_space_status == 'UNRESOLVED_OUTER_BOUND'][['industry','quarter','witnessed_possible','outer_possible']].to_string(index=False))


industry quarter sample_possible     witnessed_possible
      기계  2026Q1   OBSERVE|CHECK OBSERVE|CHECK|PRIORITY
     비금속  2022Q4           CHECK         CHECK|PRIORITY
     비금속  2023Q3           CHECK         CHECK|PRIORITY
    전기전자  2022Q4           CHECK         CHECK|PRIORITY
      철강  2025Q3         OBSERVE          OBSERVE|CHECK

미해결 행
industry quarter witnessed_possible         outer_possible
    목재종이  2023Q1              CHECK         CHECK|PRIORITY
    운송장비  2022Q2              CHECK         CHECK|PRIORITY
    운송장비  2025Q1              CHECK          OBSERVE|CHECK
    전기전자  2022Q4     CHECK|PRIORITY OBSERVE|CHECK|PRIORITY
    전기전자  2025Q1              CHECK          OBSERVE|CHECK
      철강  2022Q4              CHECK          OBSERVE|CHECK


## 3. 개정 안정성

같은 시나리오 안에서 모형들을 비교한다. 4분기 블록의 100%는 donor 블록 하나의 반복으로 생긴 퇴화 결과이며 채택하지 않는다. 오류 정정 제외 결과도 통상 개정분포의 타당성을 입증하지 않는다.

In [4]:
metrics = pd.read_csv(OUT/'revision_metrics.csv')
print(metrics[metrics.metric == 'point_retention'][['revision','model','all','disc','action_retention']].to_string(index=False))


                       revision                model      all     disc  action_retention
             legacy_independent           v1.0_crisp 0.909766 0.851236          0.942547
             legacy_independent D_small_indifference 0.902578 0.851882          0.939906
             legacy_independent               count2 0.901672 0.846826          0.939906
             legacy_independent     grouped_evidence 0.913656 0.856601          0.943422
             legacy_independent    literal_duration4 0.912578 0.853680          0.938578
               consistent_cells           v1.0_crisp 0.907734 0.849579          0.940047
               consistent_cells D_small_indifference 0.900937 0.850758          0.938406
               consistent_cells               count2 0.899656 0.845028          0.937672
               consistent_cells     grouped_evidence 0.912641 0.856067          0.941312
               consistent_cells    literal_duration4 0.912125 0.852865          0.937406
                 join

## 4. 최신 분기에 무엇을 확인할 것인가

순위가 불확실하면 공동 점검군으로 남긴다. 파라미터 범위와 자료 개정 스트레스 범위를 섞어 확률로 표현하지 않는다.

In [5]:
show('action_latest', ['industry','q1_state','g1','g2','g4','parameter_stages','revision_scenario_stages','action_group','next_check'])


industry q1_state     g1        g2   g4 parameter_stages revision_scenario_stages action_group                                                                 next_check
      기계       S4 3979.0  6.339722  2.0   CHECK|PRIORITY           CHECK|PRIORITY   불확실 공동 점검군                     수주·가동·기업 수·휴업·고용조정 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
      기타       S1    0.0  0.000000  0.0          OBSERVE                  OBSERVE       현재 관찰군                                   미충원·숙련수요·증가 지속가능성을 확인한다(현재 자료로 단정하지 않음).
    목재종이       S4   48.0 10.084034  2.0   CHECK|PRIORITY   OBSERVE|CHECK|PRIORITY   불확실 공동 점검군                     수주·가동·기업 수·휴업·고용조정 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
     비금속       S2    2.0  1.081081 19.0    OBSERVE|CHECK   OBSERVE|CHECK|PRIORITY   불확실 공동 점검군               자동화·외주화·생산성·인력부족·직무구조 변화 가능성을 질문으로 확인한다(원인 후보이며 입증된 원인이 아님).
    석유화학       S3    0.0  0.000000  0.0          OBSERVE                  OBSERVE       현재 관찰군                선제채용·신규기업·고용조정 시차·생산 시차 가능성을 질문으로 확인한다(원

## 5. 실패를 포함한 실험 기록

In [6]:
show('experiments', ['실험','목적','관측 결과','판단'])


 실험                                목적                                                    관측 결과                                  판단
E00                    원자료→마스터→국면 재실행                                      5종 CSV 수치 허용오차 내 일치                                  유지
E01                        수정 전 실행 재현 447, I3=.8928177, Jaccard=.9301498/.9477292; 18/18 C3 미달                        legacy 기록 보존
E02        같은 원계열 재사용·비교 가능한 과거 고용 이력                              g4 40행 변경; lag 값 일관성 테스트 통과                        계산 정합성 수정 채택
E03                         고용 게이트 제거                                       1080000개 배정에서 변화 0             현재 공간에서는 중복; 보호 의도는 문서화
E04               VRC4 제거·지속기간 2/4 비교                                   1745→447; 2/4 경계 동일 동작                규범 의존성 공개, 자동 승인 안 함
E05   연속공간 LP 외부범위+feasible witnesses                          154/160 집합 확인, 6행 미해결, 표본 누락 5건            표본 필연 용어 폐기; 미해결은 보수적 범위
E06         양립공간으로 I3·Jaccard 평가대상 일치                             무교란 양립표본 포함률=1; 교

## 6. 검증 기록

In [7]:
print((OUT/'raw_rebuild/comparison.json').read_text(encoding='utf-8'))
replay = OUT/'replay_verification.json'
print(replay.read_text(encoding='utf-8') if replay.exists() else '전체 재실행 대조는 진행 중입니다.')


[
  {
    "file": "changwon_industry_master",
    "rows_before": 340,
    "rows_after": 340,
    "match": true,
    "note": "match within numeric tolerance"
  },
  {
    "file": "changwon_total_master",
    "rows_before": 34,
    "rows_after": 34,
    "match": true,
    "note": "match within numeric tolerance"
  },
  {
    "file": "changwon_state_panel",
    "rows_before": 180,
    "rows_after": 180,
    "match": true,
    "note": "match within numeric tolerance"
  },
  {
    "file": "changwon_state_reference_panel",
    "rows_before": 340,
    "rows_after": 340,
    "match": true,
    "note": "match within numeric tolerance"
  },
  {
    "file": "changwon_state_sensitivity_panel",
    "rows_before": 540,
    "rows_after": 540,
    "match": true,
    "note": "match within numeric tolerance"
  }
]
[
  {
    "artifact": "revision_metrics",
    "rows": 72,
    "reproduced": true
  },
  {
    "artifact": "parameter_spaces",
    "rows": 8,
    "reproduced": true
  },
  {
    "artifact": "po